# `risk_gate.py` — Playground

Manual verification notebook for the **Phase 5 risk gate** — the final check before a
Gate 5 `BUY` becomes a live order.

| Function | Status | Notes |
|---|---|---|
| `calculate_position_size(signal, portfolio_value)` | ✅ built | Quarter-Kelly, capped at `MAX_POSITION_SIZE_PCT` |
| `validate_trade(...)` | ✅ built | 5 checks: edge, open positions, daily loss, drawdown, reward:risk |

**Deliberately thin:** edge and reward:risk reuse Gate 5's output; daily loss reuses Gate 1's
`check_daily_loss_limit()` helper. This gate only adds what upstream gates don't already cover:
open-position count, the drawdown kill switch, and sizing.


In [1]:
import sys
import pathlib

risk_dir = pathlib.Path('.').resolve()
if not (risk_dir / 'risk_gate.py').exists():
    risk_dir = pathlib.Path('backend/03_risk').resolve()

if str(risk_dir) not in sys.path:
    sys.path.insert(0, str(risk_dir))

from risk_gate import calculate_position_size, validate_trade

mock_signal = {
    'passed': True,
    'decision': 'BUY',
    'win_probability': 0.55,
    'expected_value': 0.10,
    'trade_levels': {'entry': 875.50, 'stop': 857.05, 'target': 912.40, 'reward_risk': 2.0},
}


---
## Happy path — sizing + approval

100% within limits: 1 open position, flat P&L, no drawdown.


In [2]:
calculate_position_size(mock_signal, portfolio_value=100_000)

{'position_pct': 0.08,
 'position_value': 8000.0,
 'shares': 9,
 'full_kelly_pct': 0.325}

In [3]:
validate_trade('NVDA', mock_signal, portfolio_value=100_000, daily_pnl=0,
              open_positions_count=1, drawdown_pct=0.01)

[risk] NVDA: APPROVED — 9 shares ($8,000.00, 8.0% of portfolio)


{'approved': True,
 'reject_reason': None,
 'checks': {'edge': {'passed': True, 'expected_value': 0.1},
  'open_positions': {'passed': True, 'count': 1, 'max': 5},
  'drawdown': {'passed': True, 'drawdown_pct': 0.01, 'max': 0.08},
  'reward_risk': {'passed': True, 'reward_risk': 2.0},
  'daily_loss': {'passed': True, 'loss_pct': 0.0}},
 'position': {'position_pct': 0.08,
  'position_value': 8000.0,
  'shares': 9,
  'full_kelly_pct': 0.325}}

---
## Parameter variations — weaker edge shrinks the position

Lower `win_probability` → smaller full-Kelly fraction → smaller position (still capped
by `MAX_POSITION_SIZE_PCT` on the upside).


In [4]:
weak_signal = {**mock_signal, 'win_probability': 0.42}
calculate_position_size(weak_signal, portfolio_value=100_000)

{'position_pct': 0.0325,
 'position_value': 3250.0,
 'shares': 3,
 'full_kelly_pct': 0.13}

---
## Failure path — each check can independently reject the trade


In [5]:
from config import MAX_OPEN_POSITIONS, MAX_DRAWDOWN_PCT

print(validate_trade('NVDA', mock_signal, portfolio_value=100_000, daily_pnl=0,
                      open_positions_count=MAX_OPEN_POSITIONS, drawdown_pct=0.01)['reject_reason'])
print(validate_trade('NVDA', mock_signal, portfolio_value=100_000, daily_pnl=0,
                      open_positions_count=1, drawdown_pct=MAX_DRAWDOWN_PCT)['reject_reason'])
print(validate_trade('NVDA', mock_signal, portfolio_value=100_000, daily_pnl=-3_500,
                      open_positions_count=1, drawdown_pct=0.01)['reject_reason'])


[risk] NVDA: REJECTED — open_positions check failed
open_positions
[risk] NVDA: REJECTED — drawdown check failed
drawdown
[risk] NVDA: REJECTED — daily_loss check failed
daily_loss


---
## Free-play
